# SFINCS — NJ Sandy: results & methodology viewer

A **visualization-only** companion to the build/run pipeline. The model is built
and run by the experiment harness (`run_experiments.py` + the `nj_sfincs`
package); this notebook just **opens a finished run read-only and plots it**, so
you can show your advisor any methodological choice on demand — the elevation
model, the grid, the mask, each forcing, the flood map, and the validation
against the Sandy Hook gauge / USGS High Water Marks / the FEMA MOTF extent.

The last section compares every wave experiment side by side.

> Pick which run to view by setting **`EXP`** in the setup cell. It falls back to
> the reference `model/` build if that experiment hasn't been run yet.

## Setup

In [ ]:
# Import the viz stack up top (this also primes PROJ before hydromt loads).
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# Make the nj_sfincs package importable from notebooks/.
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from hydromt_sfincs import SfincsModel
from nj_sfincs import plots, validate
from nj_sfincs.config import EXPERIMENTS

# ── Choose the run to view ───────────────────────────────────────────────────
EXP = "snapwave_tuned"  # any of: baseline_no_waves, wind_waves, snapwave_tuned, igwaves, igwaves_wind, wavemaker
exp_dir = ROOT / "experiments" / EXP
if not exp_dir.exists():
    exp_dir = ROOT / "model"  # fall back to the reference build
    print(f"experiments/{EXP} not found — showing the reference model/ build")
print("viewing:", exp_dir)


In [ ]:
# Open the run twice: `sf` (read mode, for the build + forcing inputs) and
# `mod` (for the solver output), and downscale the flood map once.
DATA_LIBS = [str(ROOT / "data" / "data_catalog.yml")]
sf = SfincsModel(str(exp_dir), data_libs=DATA_LIBS, mode="r")
sf.read()   # repopulates grid + every forcing component from disk

mod, da_hmax, da_dep = validate.load_floodmap(exp_dir)
print("output vars:", list(mod.output.data.keys()))

---
## Methodology — the static build

These document the modeling choices: the quadtree grid, the merged topobathy,
and the active/boundary mask.

### The quadtree grid

In [ ]:
plots.plot_grid(sf);

### Topobathy (interactive) — pan/zoom the dunes, inlets, dredged channels

In [ ]:
plots.plot_topobathy(sf)

### The mask — active interior, water-level boundary, outflow

In [ ]:
plots.plot_mask(sf);

---
## Methodology — the forcing

The compound drivers, read back from the written model: the observed surge
boundary, ERA5 wind/pressure, AORC rainfall, and USGS river discharge.

### Surge boundary (NOAA CO-OPS)

In [ ]:
plots.plot_surge(sf);

### Wind + pressure (ERA5)

In [ ]:
plots.plot_wind_pressure(sf);

### Rainfall (NOAA AORC)

In [ ]:
plots.plot_rain(sf);

### River discharge (USGS)

In [ ]:
plots.plot_discharge(sf);

---
## Results

The SnapWave field (if this run has waves), the downscaled flood map, and the
three validations.

### SnapWave Hm0 at peak — look for a lee behind Sandy Hook

In [ ]:
res = plots.plot_wave_field(mod)
if res is None:
    print("no wave output for this run (waves off, or no hm0)")

### Maximum flood depth

In [ ]:
plots.plot_floodmap(mod, da_hmax);

### Validation 1 — Sandy Hook gauge (temporal)

In [ ]:
m = validate.gauge_peak_error(mod)
print(f"observed peak: {m['gauge_obs_peak_m']:.2f} m | modeled peak: "
      f"{m['gauge_mod_peak_m']:.2f} m | error: {m['gauge_peak_err_m']:+.2f} m")

### Validation 2 — USGS High Water Marks (spatial)

In [ ]:
plots.plot_hwm_scatter(da_hmax, da_dep);

In [ ]:
plots.plot_hwm_residual_map(mod, da_hmax, da_dep);

### Validation 3 — FEMA MOTF extent (CSI / POD / FAR)

In [ ]:
plots.plot_motf(da_hmax, da_dep);

---
## Compare the wave experiments

Reads `experiments/metrics.csv` (written by `run_experiments.py`). Higher CSI /
POD and lower FAR / HWM-RMSE are better; `shb_hm0` is the SnapWave Hm0 in the
Sandy Hook Bay lee — the "did waves reach the bay?" number.

A self-contained **`experiments/report.html`** with the same table + flood-map
thumbnails is written by the runner — that's the file to email your advisor.

In [ ]:
metrics_csv = ROOT / "experiments" / "metrics.csv"
if metrics_csv.exists():
    metrics = pd.read_csv(metrics_csv, index_col=0)
    display(metrics.round(3))
else:
    metrics = None
    print("No experiments/metrics.csv yet — run:  python run_experiments.py")

In [ ]:
if metrics is not None:
    plots.plot_experiment_comparison(metrics, ROOT / "experiments" / "floodmaps");